<a href="https://www.kaggle.com/code/qwerty29544/cv-1-bloodmnist-excercise?scriptVersionId=365946906" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Самостоятельная работа. CNN для BloodMNIST

Цель работы — обучить свёрточную сеть определять тип клетки по цветному изображению.
Модель соберите через `nn.Sequential`.
Подготовку пакетов, обучение, валидацию и тестирование выполните самостоятельно.

## Набор данных

BloodMNIST содержит 17 092 RGB-изображения восьми типов клеток крови [1, 2].
В обучающей части 11 959 изображений, в валидационной — 1712, в тестовой — 3421.
Исходное разбиение сохраняем.

Используем версию $28\times28$.
Для входа $32\times32$ можно добавить по два пикселя рамки с каждой стороны.

В каждой таблице столбец `image` хранит массив формы `(28, 28, 3)` с яркостями от 0 до 255.
Столбец `label` содержит номер класса от 0 до 7.

Набор предназначен для исследований и обучения, а не для клинического применения [1].

## Импорт

In [1]:
from copy import deepcopy
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from IPython.display import display

## Данные

Прочитаем исходный архив MedMNIST с Zenodo [2].
Сохраним изображения в трёх отдельных таблицах.
Названия классов расположены в порядке их номеров.

In [2]:
archive_path = "/kaggle/input/datasets/qwerty29544/medmnist-13-10-2025-study-dump/bloodmnist.npz"

with np.load(archive_path, allow_pickle=False) as archive:
    df_train = pd.DataFrame({
        "image": list(archive["train_images"]),
        "label": archive["train_labels"].reshape(-1),
    })
    df_valid = pd.DataFrame({
        "image": list(archive["val_images"]),
        "label": archive["val_labels"].reshape(-1),
    })
    df_test = pd.DataFrame({
        "image": list(archive["test_images"]),
        "label": archive["test_labels"].reshape(-1),
    })

class_names = [
    "Базофил", "Эозинофил", "Эритробласт", "Незрелый гранулоцит",
    "Лимфоцит", "Моноцит", "Нейтрофил", "Тромбоцит",
]

## 1. Предпросмотр данных

Покажите несколько строк `df_train`, размеры трёх таблиц и форму одного изображения.
Проверьте диапазон пикселей и тип меток.
Выведите число обучающих изображений каждого класса.
Постройте столбчатый график этих количеств.

Покажите не меньше двух обучающих изображений каждого класса.
Подпишите названия клеток.
Достаточно таблиц и графиков, письменный вывод не требуется.

Подсказка. Массив одного изображения можно получить через `df_train.iloc[index]["image"]`.
Для `imshow` у цветного изображения каналы должны оставаться на последней оси.
Не превращайте RGB в серое изображение.

## 2. Dataset и DataLoader

### 2.1. Подготовка одного объекта

Напишите собственный класс `Dataset` для одной таблицы.
Объект должен возвращать изображение `float32` и метку `long`.
Перенесите каналы на первую ось через `permute(2, 0, 1)`.
Разделите яркости на 255 один раз.

Выберите размер входа 28 или 32 и используйте его во всех частях.
Для размера 32 дополните изображение рамкой по два пикселя с каждой стороны.
Результат одного объекта должен иметь форму `(3, 28, 28)` или `(3, 32, 32)`.
Метка одного объекта должна быть скаляром.

Подсказка. `reshape` не заменяет перестановку осей.
После перестановки рамку можно добавить через `F.pad(image, (2, 2, 2, 2))`.
Внутри `Dataset` достаточно методов `__init__`, `__len__` и `__getitem__`.

### 2.2. Пакеты для трёх частей

Создайте три объекта `Dataset` по готовым таблицам.
Не объединяйте части и не делите их заново.
Составьте загрузчики для обучения, валидации и теста.
Для обучения включите перемешивание.
Для двух остальных частей оставьте исходный порядок.

Задайте размер пакета 64 или 128.
Выведите формы и типы одного обучающего пакета.
Подготовьте отдельный загрузчик без перемешивания для итоговой оценки обучающей части.
Зафиксируйте генератор случайных чисел.

Подсказка. Вход модели должен иметь форму `(N, 3, H, W)`, метки — `(N,)`.
Для начала достаточно `num_workers=0`.

## 3. Функция потерь

Выберите функцию потерь для классификации с одним правильным классом из восьми.
Создайте её объект.
Укажите формы предсказаний и целевых меток в коротком комментарии рядом с кодом.

Подсказка. Для `CrossEntropyLoss` передаются логиты, а не вероятности [3].
Метки не нужно превращать в one-hot векторы.
Начните с одинакового веса всех объектов.

## 4. Архитектура через Sequential

### 4.1. Свёрточная часть и классификатор

Соберите модель из двух или трёх свёрточных блоков.
Включите `Conv2d`, `BatchNorm2d`, `ReLU` и уменьшение карт через pooling.
Для первого свёрточного слоя задайте три входных канала.
Число выходных карт выберите самостоятельно.

После свёрточной части добавьте `Flatten` и полносвязный классификатор.
Последний слой должен возвращать восемь логитов.
Softmax в модель не добавляйте.
Выберите устройство и перенесите на него модель.

Подсказка. После окна pooling 2 с шагом 2 размер 28 станет 14, затем 7.
Размер 32 станет 16, затем 8.
Перед BatchNorm у свёртки можно отключить `bias`.
Для фиксированного выхода свёрточной части также можно использовать `AdaptiveAvgPool2d`.

### 4.2. Формы и число параметров

Проведите небольшой пакет через модель без обучения.
Покажите форму выхода после каждого слоя.
Проверьте, что один объект даёт ровно восемь логитов.
Посчитайте общее число обучаемых параметров.

Подсказка. Перед просмотром форм переключите модель в `eval()`.
Иначе BatchNorm обновит бегущие статистики на пробном пакете.
Используйте `torch.no_grad()` и цикл по слоям `Sequential`.

## 5. Оптимизатор

Выберите оптимизатор и задайте шаг обучения.
Передайте ему параметры созданной модели.
Задайте число эпох.

Подсказка. Можно начать с `Adam` и шага $10^{-3}$.
Оптимизатор создаётся один раз перед обучением, а не заново на каждой эпохе.
Параметры эксперимента подбирайте по валидационной части.

## 6. Обучение с валидацией

### 6.1. История и лучшее состояние

Подготовьте списки для обучающей и валидационной ошибки.
Подготовьте списки для accuracy на обеих частях.
Создайте переменные для лучшей валидационной ошибки, номера эпохи и копии состояния модели.

Подсказка. Начальное лучшее значение ошибки можно задать как `float("inf")`.
Для сохранения состояния используйте `deepcopy(model.state_dict())`.
Обычная ссылка на `state_dict` не заменяет независимую копию.

### 6.2. Цикл по эпохам и пакетам

Напишите внешний цикл по эпохам.
В начале каждой эпохи включите `model.train()`.
Внутри обучающего цикла перенесите изображения и метки на устройство.
Обнулите градиенты, вычислите логиты и ошибку, выполните `backward()` и шаг оптимизатора.

После обучающего цикла включите `model.eval()`.
Пройдите валидационные пакеты без записи графа и без обновления весов.
Сохраните ошибку и accuracy обеих частей.
Обновляйте лучшее состояние только по валидационной ошибке.
Выводите одну короткую строку на эпоху.

Подсказка. Для средней ошибки умножайте ошибку пакета на число объектов в нём.
В конце делите сумму на размер соответствующей части.
В `BatchNorm2d` поведение зависит от `train()` и `eval()` [4].
Одного `no_grad()` для валидации недостаточно.

## 7. Собственный цикл тестирования

Восстановите лучшее состояние модели.
Переключите её в `eval()`.
Напишите цикл по тестовому загрузчику без вычисления градиентов.
Соберите правильные метки, предсказанные классы и вероятности восьми классов.
Сохраните их в порядке таблицы `df_test`.

Посчитайте среднюю тестовую ошибку.
Не меняйте веса и статистики BatchNorm на тестовых изображениях.
Не выбирайте по тесту лучшую эпоху.

Подсказка. Класс получается через `argmax(dim=1)`.
Для вероятностей примените `softmax(dim=1)` к логитам.
Перед преобразованием в NumPy перенесите результаты на CPU.

## 8. Метрики и графики обучения

### 8.1. Итоговая оценка train и test

Сделайте отдельный проход лучшей модели по обучающей части в режиме оценки.
Посчитайте accuracy и macro-F1 на train и test.
Сведите значения в одну таблицу.
Покажите precision, recall, F1 и число объектов для каждого тестового класса.

Подсказка. Используйте все восемь меток и одинаковый порядок названий.
Для отсутствующих предсказаний класса можно указать `zero_division=0`.
Macro-F1 полезно сопоставить с accuracy, когда классы имеют разные размеры.

### 8.2. Кривые по эпохам

На одном графике покажите обучающую и валидационную ошибку.
На другом — обучающую и валидационную accuracy.
Подпишите оси и добавьте легенды.
Отметьте эпоху, состояние которой было выбрано.

Подсказка. Обучающая метрика внутри эпохи получена во время изменения весов.
Она может отличаться от отдельной оценки лучшей модели на train.

## 9. Визуализации ответов и карт признаков

### 9.1. Матрица ошибок

Постройте матрицу ошибок на тестовой части.
Подпишите все восемь классов.
Дополнительно покажите матрицу, нормированную по строкам.

Подсказка. Строка соответствует правильному классу.
Сумма нормированной строки равна единице.
Число изображений каждого класса возьмите из итогового отчёта по метрикам.

### 9.2. Верные ответы и ошибки

Покажите несколько правильно и несколько неправильно классифицированных тестовых изображений.
Подпишите правильный класс, ответ модели и максимальную предсказанную вероятность.
Для одного ошибочного изображения постройте столбчатый график вероятностей всех классов.

Подсказка. Маска ошибки имеет вид `predictions != targets`.
Сохраняйте соответствие между номером ответа и строкой `df_test`.
Высокая максимальная вероятность сама по себе не гарантирует правильный ответ.

### 9.3. Промежуточные карты

Возьмите одно тестовое изображение и покажите его отдельно.
Получите выход первого блока после ReLU, но до pooling.
Покажите не меньше шести карт с номерами каналов.
Используйте общую шкалу значений для этих карт.

Подсказка. В `Sequential` можно выбрать начальный фрагмент через срез.
Границу среза определите по своей архитектуре.
Перед получением карт включите `eval()` и отключите запись графа.
Карта признака — это не вероятность класса и не готовая карта объяснения решения.

## Источники

[1] [MedMNIST. Описание набора и назначение данных](https://medmnist.com/).

[2] [MedMNIST. Официальные сведения о BloodMNIST, метках и файлах](https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py).

[3] [PyTorch. CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html).

[4] [PyTorch. BatchNorm2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html).